# NB00 · Configuración y descarga desde Zenodo

**Proyecto:** Palma africana con visión artificial  
**Objetivo de esta fase:** crear la estructura persistente en Google Drive, consultar el registro oficial de Zenodo, descargar el archivo original y verificar su integridad.

Dataset inicial:

- **Outdoor Tenera Oil Palm Fruit Image: FFB Dataset**
- Zenodo: https://zenodo.org/records/11114885
- DOI: https://doi.org/10.5281/zenodo.11114885
- Archivo publicado: aproximadamente 1,2 GB

> Este notebook no entrena todavía ningún modelo. Primero conserva y verifica los datos originales.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import platform
import requests
import sys
from urllib.parse import unquote

PROJECT_ROOT = Path('/content/drive/MyDrive/PALMA_AFRICANA_YOLO')
DIRS = {
    'docs': PROJECT_ROOT / '00_DOCUMENTACION',
    'raw': PROJECT_ROOT / '01_DATOS_ORIGINALES',
    'processed': PROJECT_ROOT / '02_DATOS_PROCESADOS',
    'splits': PROJECT_ROOT / '03_SPLITS',
    'notebooks': PROJECT_ROOT / '04_NOTEBOOKS',
    'models': PROJECT_ROOT / '05_MODELOS',
    'results': PROJECT_ROOT / '06_RESULTADOS',
    'videos': PROJECT_ROOT / '07_VIDEOS',
    'book': PROJECT_ROOT / '08_MATERIAL_LIBRO',
}
for path in DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

print('Python:', sys.version.split()[0])
print('Sistema:', platform.platform())
print('Proyecto:', PROJECT_ROOT)
print('Carpetas verificadas:', len(DIRS))


In [ ]:
RECORD_ID = '11114885'
API_URL = f'https://zenodo.org/api/records/{RECORD_ID}'

response = requests.get(API_URL, timeout=60)
response.raise_for_status()
metadata = response.json()

metadata_path = DIRS['docs'] / f'zenodo_{RECORD_ID}_metadata.json'
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')

files_block = metadata.get('files', {})
if isinstance(files_block, dict) and isinstance(files_block.get('entries'), dict):
    files = list(files_block['entries'].values())
elif isinstance(files_block, list):
    files = files_block
else:
    files = []

print('Título:', metadata.get('metadata', {}).get('title', 'No disponible'))
print('Archivos encontrados:', len(files))
for i, item in enumerate(files, 1):
    print(f"{i}. {item.get('key')} | {item.get('size', 0)/1024**3:.3f} GB | {item.get('checksum')}")

if not files:
    raise RuntimeError('La API no devolvió archivos. Revise el registro antes de continuar.')


In [ ]:
def md5_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.md5()
    with open(path, 'rb') as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def download_file(item):
    key = item.get('key', 'zenodo_file.bin')
    filename = Path(unquote(key)).name
    destination = DIRS['raw'] / filename
    expected_size = int(item.get('size', 0) or 0)
    checksum = item.get('checksum', '')
    expected_md5 = checksum.split(':', 1)[1] if checksum.startswith('md5:') else None

    links = item.get('links', {})
    url = links.get('content') or links.get('self')
    if not url:
        raise RuntimeError(f'No se encontró enlace de descarga para {key}')

    if destination.exists() and (not expected_size or destination.stat().st_size == expected_size):
        print('El archivo ya existe con el tamaño esperado:', destination)
    else:
        tmp = destination.with_suffix(destination.suffix + '.part')
        downloaded = tmp.stat().st_size if tmp.exists() else 0
        headers = {'Range': f'bytes={downloaded}-'} if downloaded else {}
        mode = 'ab' if downloaded else 'wb'
        with requests.get(url, stream=True, timeout=120, headers=headers) as r:
            if downloaded and r.status_code == 200:
                downloaded = 0
                mode = 'wb'
            r.raise_for_status()
            total = expected_size or int(r.headers.get('Content-Length', 0)) + downloaded
            with open(tmp, mode) as out:
                for chunk in r.iter_content(chunk_size=8 * 1024 * 1024):
                    if chunk:
                        out.write(chunk)
                        downloaded += len(chunk)
                        if total:
                            print(f'Descarga: {100*downloaded/total:6.2f}% ', end='\r')
        print()
        tmp.replace(destination)

    if expected_size and destination.stat().st_size != expected_size:
        raise RuntimeError('El tamaño descargado no coincide con el publicado por Zenodo.')

    if expected_md5:
        observed_md5 = md5_file(destination)
        if observed_md5.lower() != expected_md5.lower():
            raise RuntimeError(f'MD5 incorrecto para {destination.name}')
        print('MD5 verificado:', observed_md5)
    return destination


downloaded_files = [download_file(item) for item in files]
print('\nDescarga completada:')
for path in downloaded_files:
    print('-', path, f'({path.stat().st_size/1024**3:.3f} GB)')


## Resultado esperado

Al finalizar deben existir:

1. El ZIP original dentro de `01_DATOS_ORIGINALES`.
2. El metadato oficial `zenodo_11114885_metadata.json` dentro de `00_DOCUMENTACION`.
3. Una confirmación de que el tamaño y el MD5 coinciden con Zenodo.

Después ejecute **NB01_PALMA_Inventario_Auditoria.ipynb**.
